In [1]:
import sounddevice as sd
import numpy as np
import tkinter as tk
from tkinter import ttk
import threading
import time

# Configure sounddevice for low latency
sd.default.latency = 'low'

class MouseClickToneGenerator:
    """Class to generate and play tones on mouse clicks using output 3/4"""
    def __init__(self, frequency=1000, duration=0.1, sample_rate=44100):
        self.frequency = frequency    # Tone frequency in Hz
        self.duration = duration      # Tone duration in seconds
        self.sample_rate = sample_rate
        self.output_34_id = None
        self.stream = None
        self.active = False  # Flag to track if we're currently playing
        
        # Generate the tone once and reuse it
        self.tone_data = self._generate_tone()
        
        # Find output 3/4 device
        self._find_output_device()
        
        # Try to open the stream once at initialization
        self._open_stream()
    
    def _generate_tone(self):
        """Generate the tone signal"""
        t = np.linspace(0, self.duration, int(self.sample_rate * self.duration), endpoint=False)
        tone = 0.5 * np.sin(2 * np.pi * self.frequency * t).astype(np.float32)
        
        # Apply short fade in/out to prevent clicks
        fade_samples = int(0.01 * self.sample_rate)  # 10ms fade
        fade_in = np.linspace(0, 1, fade_samples)
        fade_out = np.linspace(1, 0, fade_samples)
        
        tone[:fade_samples] *= fade_in
        tone[-fade_samples:] *= fade_out
        
        # Create stereo version (for output to both channels in device)
        stereo_tone = np.column_stack((tone, tone))
        return stereo_tone
    
    def _find_output_device(self):
        """Find the Output 3/4 device"""
        devices = sd.query_devices()
        
        # First try to use the default output device as a fallback
        self.output_34_id = sd.default.device[1]
        
        # Then look for Komplete Audio ASIO devices or similar
        for i, device in enumerate(devices):
            if device['max_output_channels'] > 0:
                name = device['name'].lower()
                # Look for non-ASIO devices first as they're more compatible
                if "output" in name and "3/4" in name:
                    self.output_34_id = i
                    print(f"Found Output 3/4 device: ID {i} - {device['name']}")
                    return
        
        print(f"Using default output device ID {self.output_34_id}")
    
    def _open_stream(self):
        """Open a persistent audio stream"""
        try:
            # Create a silent buffer for when not playing
            self.silent_buffer = np.zeros((1024, 2), dtype=np.float32)
            
            # Create the stream with a callback
            self.stream = sd.OutputStream(
                device=self.output_34_id,
                channels=2,
                callback=self._stream_callback,
                samplerate=self.sample_rate,
                blocksize=256,  # Small blocksize for low latency
                dtype='float32'
            )
            
            # Start the stream
            self.stream.start()
            print(f"Audio stream opened successfully on device {self.output_34_id}")
            
            # Create a position variable to track where we are in the tone
            self.position = 0
            
        except Exception as e:
            print(f"Error opening audio stream: {e}")
            self.stream = None
    
    def _stream_callback(self, outdata, frames, time, status):
        """Callback for the persistent audio stream"""
        if status:
            print(f"Stream callback status: {status}")
            
        if not self.active:
            # If not active, just fill with silence
            outdata.fill(0)
            return
            
        # We're playing the tone
        if self.position + frames > len(self.tone_data):
            # We've reached the end of the tone
            remaining = len(self.tone_data) - self.position
            if remaining > 0:
                outdata[:remaining] = self.tone_data[self.position:self.position+remaining]
            outdata[remaining:] = 0
            self.position = 0
            self.active = False  # Done playing
        else:
            # Still playing the tone
            outdata[:] = self.tone_data[self.position:self.position+frames]
            self.position += frames
    
    def play_tone(self, event=None):
        """Play the tone on mouse click event"""
        if self.stream is None or not self.stream.active:
            # Try to reopen the stream if it's not open
            self._open_stream()
            if self.stream is None:
                print("Cannot play tone: no active audio stream")
                return
        
        # Reset position and set active flag
        self.position = 0
        self.active = True


class MouseClickToneApp:
    """Simple app to demonstrate the tone generator"""
    def __init__(self, root):
        self.root = root
        self.root.title("Mouse Click Tone Generator")
        self.root.geometry("400x200")
        
        # Create tone generator
        self.tone_generator = MouseClickToneGenerator()
        
        # Create UI
        frame = ttk.Frame(root, padding="20")
        frame.pack(fill=tk.BOTH, expand=True)
        
        title = ttk.Label(frame, text="Mouse Click Tone Generator", font=("Arial", 14, "bold"))
        title.pack(pady=10)
        
        info = ttk.Label(frame, text="Click anywhere to generate a tone")
        info.pack(pady=5)
        
        test_btn = ttk.Button(frame, text="Test Tone", command=self.tone_generator.play_tone)
        test_btn.pack(pady=10)
        
        # Add a status message
        self.status_var = tk.StringVar(value="Ready")
        status_label = ttk.Label(frame, textvariable=self.status_var)
        status_label.pack(pady=5)
        
        # Bind mouse clicks
        for button in ("<Button-1>", "<Button-2>", "<Button-3>"):
            root.bind(button, self.tone_generator.play_tone)
            
        # Ensure clean closing
        root.protocol("WM_DELETE_WINDOW", self.on_closing)
        
    def on_closing(self):
        """Handle window closing"""
        # Close the stream if it exists
        if hasattr(self.tone_generator, 'stream') and self.tone_generator.stream:
            self.tone_generator.stream.stop()
            self.tone_generator.stream.close()
        
        # Stop any active streams
        try:
            sd.stop()
        except:
            pass
        
        self.root.destroy()


# Function to add mouse click tones to an existing window
def add_click_tones_to_window(window):
    """Add click tone functionality to any existing tkinter window"""
    tone_generator = MouseClickToneGenerator()
    
    # Bind mouse clicks to the tone function
    for button in ("<Button-1>", "<Button-2>", "<Button-3>"):
        window.bind(button, tone_generator.play_tone)
        
    # Return the generator so it can be kept alive
    return tone_generator


# Standalone application
if __name__ == "__main__":
    root = tk.Tk()
    app = MouseClickToneApp(root)
    root.mainloop()

Found Output 3/4 device: ID 7 - Output 3/4 (2- Komplete Audio 6
Audio stream opened successfully on device 7
Stream callback status: output underflow
Stream callback status: output underflow
Stream callback status: output underflow
Stream callback status: output underflow
Stream callback status: output underflow
Stream callback status: output underflow
Stream callback status: output underflow
Stream callback status: output underflow
Stream callback status: output underflow


KeyboardInterrupt: 

In [2]:
import sounddevice as sd
import numpy as np
import tkinter as tk
from tkinter import ttk
import threading
import time
import sys

# For Windows global mouse hook
import ctypes
from ctypes import wintypes
import win32con

# Configure sounddevice for low latency
sd.default.latency = 'low'

class MouseClickToneGenerator:
    """Class to generate and play tones on mouse clicks using output 3/4"""
    def __init__(self, frequency=1000, duration=0.1, sample_rate=44100, amplitude=0.25):
        self.frequency = frequency    # Tone frequency in Hz
        self.duration = duration      # Tone duration in seconds
        self.sample_rate = sample_rate
        self.amplitude = amplitude    # Reduced to 0.25 (half of previous 0.5)
        self.output_34_id = None
        self.stream = None
        self.active = False  # Flag to track if we're currently playing
        
        # Generate the tone once and reuse it
        self.tone_data = self._generate_tone()
        
        # Find output 3/4 device
        self._find_output_device()
        
        # Try to open the stream once at initialization
        self._open_stream()
    
    def _generate_tone(self):
        """Generate the tone signal"""
        t = np.linspace(0, self.duration, int(self.sample_rate * self.duration), endpoint=False)
        tone = self.amplitude * np.sin(2 * np.pi * self.frequency * t).astype(np.float32)
        
        # Apply short fade in/out to prevent clicks
        fade_samples = int(0.01 * self.sample_rate)  # 10ms fade
        fade_in = np.linspace(0, 1, fade_samples)
        fade_out = np.linspace(1, 0, fade_samples)
        
        tone[:fade_samples] *= fade_in
        tone[-fade_samples:] *= fade_out
        
        # Create stereo version (for output to both channels in device)
        stereo_tone = np.column_stack((tone, tone))
        return stereo_tone
    
    def _find_output_device(self):
        """Find the Output 3/4 device"""
        devices = sd.query_devices()
        
        # First try to use the default output device as a fallback
        self.output_34_id = sd.default.device[1]
        
        # Then look for Komplete Audio ASIO devices or similar
        for i, device in enumerate(devices):
            if device['max_output_channels'] > 0:
                name = device['name'].lower()
                # Look for non-ASIO devices first as they're more compatible
                if "output" in name and "3/4" in name:
                    self.output_34_id = i
                    print(f"Found Output 3/4 device: ID {i} - {device['name']}")
                    return
        
        print(f"Using default output device ID {self.output_34_id}")
    
    def _open_stream(self):
        """Open a persistent audio stream"""
        try:
            # Create a silent buffer for when not playing
            self.silent_buffer = np.zeros((1024, 2), dtype=np.float32)
            
            # Create the stream with a callback
            self.stream = sd.OutputStream(
                device=self.output_34_id,
                channels=2,
                callback=self._stream_callback,
                samplerate=self.sample_rate,
                blocksize=256,  # Small blocksize for low latency
                dtype='float32'
            )
            
            # Start the stream
            self.stream.start()
            print(f"Audio stream opened successfully on device {self.output_34_id}")
            
            # Create a position variable to track where we are in the tone
            self.position = 0
            
        except Exception as e:
            print(f"Error opening audio stream: {e}")
            self.stream = None
    
    def _stream_callback(self, outdata, frames, time, status):
        """Callback for the persistent audio stream"""
        if status:
            print(f"Stream callback status: {status}")
            
        if not self.active:
            # If not active, just fill with silence
            outdata.fill(0)
            return
            
        # We're playing the tone
        if self.position + frames > len(self.tone_data):
            # We've reached the end of the tone
            remaining = len(self.tone_data) - self.position
            if remaining > 0:
                outdata[:remaining] = self.tone_data[self.position:self.position+remaining]
            outdata[remaining:] = 0
            self.position = 0
            self.active = False  # Done playing
        else:
            # Still playing the tone
            outdata[:] = self.tone_data[self.position:self.position+frames]
            self.position += frames
    
    def play_tone(self, event=None):
        """Play the tone on mouse click event"""
        if self.stream is None or not self.stream.active:
            # Try to reopen the stream if it's not open
            self._open_stream()
            if self.stream is None:
                print("Cannot play tone: no active audio stream")
                return
        
        # Reset position and set active flag
        self.position = 0
        self.active = True
    
    def cleanup(self):
        """Clean up resources"""
        if self.stream:
            self.stream.stop()
            self.stream.close()
            self.stream = None


# Windows global mouse hook implementation
class GlobalMouseHook:
    """Class to detect mouse clicks system-wide on Windows"""
    
    # Define required Windows API structures
    class MSLLHOOKSTRUCT(ctypes.Structure):
        _fields_ = [
            ("pt", wintypes.POINT),
            ("mouseData", wintypes.DWORD),
            ("flags", wintypes.DWORD),
            ("time", wintypes.DWORD),
            ("dwExtraInfo", ctypes.c_void_p)
        ]
    
    def __init__(self, callback):
        self.callback = callback
        self.hook_id = None
        self.running = False
        
        # Define callback type for low-level mouse hook
        self.HOOKPROC = ctypes.CFUNCTYPE(
            ctypes.c_int, 
            ctypes.c_int, 
            wintypes.WPARAM, 
            wintypes.LPARAM
        )
        
        # Store instances to prevent garbage collection
        self.hook_proc = self.HOOKPROC(self.mouse_hook_proc)
        
        # Define constants
        self.WH_MOUSE_LL = 14
        self.WM_LBUTTONDOWN = 0x0201
        self.WM_RBUTTONDOWN = 0x0204
        self.WM_MBUTTONDOWN = 0x0207
    
    def mouse_hook_proc(self, n_code, w_param, l_param):
        """Mouse hook callback function"""
        if n_code >= 0:
            # Check if it's a button down message
            if w_param in (self.WM_LBUTTONDOWN, self.WM_RBUTTONDOWN, self.WM_MBUTTONDOWN):
                mouse_struct = ctypes.cast(l_param, ctypes.POINTER(self.MSLLHOOKSTRUCT)).contents
                
                # Call the user callback (tone generator)
                self.callback(None)  # No event parameter needed
        
        # Call next hook in chain
        return ctypes.windll.user32.CallNextHookEx(self.hook_id, n_code, w_param, l_param)
    
    def start(self):
        """Start the global mouse hook"""
        if self.running:
            return
            
        self.hook_id = ctypes.windll.user32.SetWindowsHookExA(
            self.WH_MOUSE_LL,
            self.hook_proc,
            ctypes.windll.kernel32.GetModuleHandleA(None),
            0
        )
        
        if not self.hook_id:
            raise Exception("Failed to install mouse hook")
            
        self.running = True
        print("Global mouse hook installed successfully")
        
        # Start message loop in a separate thread
        self.message_thread = threading.Thread(target=self._message_loop, daemon=True)
        self.message_thread.start()
    
    def _message_loop(self):
        """Windows message loop to process hook events"""
        msg = wintypes.MSG()
        while self.running:
            # Get and dispatch messages
            if ctypes.windll.user32.GetMessageA(ctypes.byref(msg), None, 0, 0) > 0:
                ctypes.windll.user32.TranslateMessage(ctypes.byref(msg))
                ctypes.windll.user32.DispatchMessageA(ctypes.byref(msg))
            time.sleep(0.001)  # Small sleep to reduce CPU usage
    
    def stop(self):
        """Stop the global mouse hook"""
        if self.running and self.hook_id:
            self.running = False
            ctypes.windll.user32.UnhookWindowsHookEx(self.hook_id)
            self.hook_id = None
            print("Global mouse hook removed")


class SystemWideMouseClickToneApp:
    """App that responds to mouse clicks system-wide"""
    def __init__(self, root):
        self.root = root
        self.root.title("System-Wide Mouse Click Tone Generator")
        self.root.geometry("400x250")
        
        # Create tone generator
        self.tone_generator = MouseClickToneGenerator(amplitude=0.25)  # Reduced amplitude
        
        # Create UI
        self._create_ui()
        
        # Create and start the global mouse hook
        try:
            self.mouse_hook = GlobalMouseHook(self.tone_generator.play_tone)
            self.mouse_hook.start()
            self.status_var.set("Global mouse hook active - listening for clicks anywhere")
        except Exception as e:
            self.status_var.set(f"Error: {str(e)}")
            print(f"Error setting up global mouse hook: {e}")
            
        # Ensure clean closing
        root.protocol("WM_DELETE_WINDOW", self.on_closing)
    
    def _create_ui(self):
        """Create the user interface"""
        frame = ttk.Frame(self.root, padding="20")
        frame.pack(fill=tk.BOTH, expand=True)
        
        title = ttk.Label(frame, text="System-Wide Mouse Click Tone Generator", 
                         font=("Arial", 12, "bold"), wraplength=350)
        title.pack(pady=10)
        
        info = ttk.Label(frame, 
                        text="This app plays a tone on output 3/4 whenever you click anywhere on your computer.",
                        wraplength=350)
        info.pack(pady=5)
        
        test_btn = ttk.Button(frame, text="Test Tone", command=self.tone_generator.play_tone)
        test_btn.pack(pady=10)
        
        # Add amplitude control
        amp_frame = ttk.Frame(frame)
        amp_frame.pack(pady=5, fill=tk.X)
        
        ttk.Label(amp_frame, text="Amplitude:").pack(side=tk.LEFT, padx=(0, 5))
        
        self.amplitude_var = tk.DoubleVar(value=0.25)
        
        def update_amplitude(event=None):
            new_amp = self.amplitude_var.get()
            self.tone_generator.amplitude = new_amp
            self.tone_generator.tone_data = self.tone_generator._generate_tone()
        
        amplitude_scale = ttk.Scale(amp_frame, from_=0.01, to=0.5, 
                                  variable=self.amplitude_var, 
                                  command=update_amplitude)
        amplitude_scale.pack(side=tk.LEFT, fill=tk.X, expand=True)
        
        # Add a status message
        self.status_var = tk.StringVar(value="Initializing...")
        status_label = ttk.Label(frame, textvariable=self.status_var, wraplength=350)
        status_label.pack(pady=10)
        
    def on_closing(self):
        """Handle window closing"""
        # Stop the mouse hook
        if hasattr(self, 'mouse_hook'):
            self.mouse_hook.stop()
        
        # Cleanup tone generator
        if hasattr(self, 'tone_generator'):
            self.tone_generator.cleanup()
        
        # Stop any active streams
        try:
            sd.stop()
        except:
            pass
        
        self.root.destroy()


# Create a system-wide mouse click detector and tone generator
def create_system_wide_tone_generator():
    """Create a tone generator that responds to system-wide mouse clicks"""
    # Create the tone generator
    tone_generator = MouseClickToneGenerator(amplitude=0.25)
    
    # Create the global mouse hook
    mouse_hook = GlobalMouseHook(tone_generator.play_tone)
    mouse_hook.start()
    
    return tone_generator, mouse_hook


# Standalone application
if __name__ == "__main__":
    if sys.platform != 'win32':
        print("This script only supports Windows for global mouse detection")
        sys.exit(1)
        
    root = tk.Tk()
    app = SystemWideMouseClickToneApp(root)
    root.mainloop()

Found Output 3/4 device: ID 7 - Output 3/4 (2- Komplete Audio 6
Audio stream opened successfully on device 7
Error setting up global mouse hook: Failed to install mouse hook
Stream callback status: output underflow
Stream callback status: output underflow
Stream callback status: output underflow


KeyboardInterrupt: 

Found Output 3/4 device: ID 7 - Output 3/4 (2- Komplete Audio 6
Audio stream opened successfully on device 7
Failed to install mouse hook: Error code 126


KeyboardInterrupt: 

In [1]:
import sounddevice as sd
import numpy as np
import threading
import time
import sys
import ctypes
from ctypes import wintypes
import win32con

# Configure sounddevice for low latency
sd.default.latency = 'low'

class MouseClickToneGenerator:
    """Class to generate and play tones on mouse clicks using output 3/4"""
    def __init__(self, frequency=1000, duration=0.1, sample_rate=44100, amplitude=0.25):
        self.frequency = frequency    # Tone frequency in Hz
        self.duration = duration      # Tone duration in seconds
        self.sample_rate = sample_rate
        self.amplitude = amplitude    # Reduced to 0.25 to keep it less intrusive
        self.output_34_id = None
        self.stream = None
        self.active = False  # Flag to track if we're currently playing
        
        # Generate the tone once and reuse it
        self.tone_data = self._generate_tone()
        
        # Find output 3/4 device
        self._find_output_device()
        
        # Try to open the stream once at initialization
        self._open_stream()
        
        print(f"Tone generator initialized with frequency {frequency}Hz, amplitude {amplitude}")
    
    def _generate_tone(self):
        """Generate the tone signal"""
        t = np.linspace(0, self.duration, int(self.sample_rate * self.duration), endpoint=False)
        tone = self.amplitude * np.sin(2 * np.pi * self.frequency * t).astype(np.float32)
        
        # Apply short fade in/out to prevent clicks
        fade_samples = int(0.01 * self.sample_rate)  # 10ms fade
        fade_in = np.linspace(0, 1, fade_samples)
        fade_out = np.linspace(1, 0, fade_samples)
        
        tone[:fade_samples] *= fade_in
        tone[-fade_samples:] *= fade_out
        
        # Create stereo version (for output to both channels in device)
        stereo_tone = np.column_stack((tone, tone))
        return stereo_tone
    
    def _find_output_device(self):
        """Find the Output 3/4 device"""
        devices = sd.query_devices()
        
        # First try to use the default output device as a fallback
        self.output_34_id = sd.default.device[1]
        
        # Then look for Komplete Audio ASIO devices or similar
        for i, device in enumerate(devices):
            if device['max_output_channels'] > 0:
                name = device['name'].lower()
                # Look for output 3/4 devices
                if "output" in name and "3/4" in name:
                    self.output_34_id = i
                    print(f"Found Output 3/4 device: ID {i} - {device['name']}")
                    return
        
        print(f"Using default output device ID {self.output_34_id}")
    
    def _open_stream(self):
        """Open a persistent audio stream"""
        try:
            # Create a silent buffer for when not playing
            self.silent_buffer = np.zeros((1024, 2), dtype=np.float32)
            
            # Create the stream with a callback
            self.stream = sd.OutputStream(
                device=self.output_34_id,
                channels=2,
                callback=self._stream_callback,
                samplerate=self.sample_rate,
                blocksize=256,  # Small blocksize for low latency
                dtype='float32'
            )
            
            # Start the stream
            self.stream.start()
            print(f"Audio stream opened successfully on device {self.output_34_id}")
            
            # Create a position variable to track where we are in the tone
            self.position = 0
            
        except Exception as e:
            print(f"Error opening audio stream: {e}")
            self.stream = None
    
    def _stream_callback(self, outdata, frames, time, status):
        """Callback for the persistent audio stream"""
        if status:
            print(f"Stream callback status: {status}")
            
        if not self.active:
            # If not active, just fill with silence
            outdata.fill(0)
            return
            
        # We're playing the tone
        if self.position + frames > len(self.tone_data):
            # We've reached the end of the tone
            remaining = len(self.tone_data) - self.position
            if remaining > 0:
                outdata[:remaining] = self.tone_data[self.position:self.position+remaining]
            outdata[remaining:] = 0
            self.position = 0
            self.active = False  # Done playing
        else:
            # Still playing the tone
            outdata[:] = self.tone_data[self.position:self.position+frames]
            self.position += frames
    
    def play_tone(self, event=None):
        """Play the tone on mouse click event"""
        if self.stream is None or not self.stream.active:
            # Try to reopen the stream if it's not open
            self._open_stream()
            if self.stream is None:
                print("Cannot play tone: no active audio stream")
                return
        
        # Reset position and set active flag
        self.position = 0
        self.active = True
        
        # Print a dot to indicate a click was detected (without cluttering the console)
        sys.stdout.write('.')
        sys.stdout.flush()
    
    def cleanup(self):
        """Clean up resources"""
        if self.stream:
            self.stream.stop()
            self.stream.close()
            self.stream = None
        print("\nTone generator cleaned up")


# Windows global mouse hook implementation
class GlobalMouseHook:
    """Class to detect mouse clicks system-wide on Windows"""
    
    # Define required Windows API structures
    class MSLLHOOKSTRUCT(ctypes.Structure):
        _fields_ = [
            ("pt", wintypes.POINT),
            ("mouseData", wintypes.DWORD),
            ("flags", wintypes.DWORD),
            ("time", wintypes.DWORD),
            ("dwExtraInfo", ctypes.c_void_p)
        ]
    
    def __init__(self, callback):
        self.callback = callback
        self.hook_id = None
        self.running = False
        
        # Define callback type for low-level mouse hook
        self.HOOKPROC = ctypes.CFUNCTYPE(
            ctypes.c_int, 
            ctypes.c_int, 
            wintypes.WPARAM, 
            wintypes.LPARAM
        )
        
        # Store instances to prevent garbage collection
        self.hook_proc = self.HOOKPROC(self.mouse_hook_proc)
        
        # Define constants
        self.WH_MOUSE_LL = 14
        self.WM_LBUTTONDOWN = 0x0201
        self.WM_RBUTTONDOWN = 0x0204
        self.WM_MBUTTONDOWN = 0x0207
    
    def mouse_hook_proc(self, n_code, w_param, l_param):
        """Mouse hook callback function"""
        if n_code >= 0:
            # Check if it's a button down message
            if w_param in (self.WM_LBUTTONDOWN, self.WM_RBUTTONDOWN, self.WM_MBUTTONDOWN):
                mouse_struct = ctypes.cast(l_param, ctypes.POINTER(self.MSLLHOOKSTRUCT)).contents
                
                # Call the user callback (tone generator)
                self.callback(None)  # No event parameter needed
        
        # Call next hook in chain
        return ctypes.windll.user32.CallNextHookEx(self.hook_id, n_code, w_param, l_param)
    
    def start(self):
        """Start the global mouse hook"""
        if self.running:
            return
            
        self.hook_id = ctypes.windll.user32.SetWindowsHookExA(
            self.WH_MOUSE_LL,
            self.hook_proc,
            ctypes.windll.kernel32.GetModuleHandleA(None),
            0
        )
        
        if not self.hook_id:
            raise Exception("Failed to install mouse hook")
            
        self.running = True
        print("Global mouse hook installed successfully")
        
        # Start message loop in a separate thread
        self.message_thread = threading.Thread(target=self._message_loop, daemon=True)
        self.message_thread.start()
    
    def _message_loop(self):
        """Windows message loop to process hook events"""
        msg = wintypes.MSG()
        while self.running:
            # Get and dispatch messages
            if ctypes.windll.user32.GetMessageA(ctypes.byref(msg), None, 0, 0) > 0:
                ctypes.windll.user32.TranslateMessage(ctypes.byref(msg))
                ctypes.windll.user32.DispatchMessageA(ctypes.byref(msg))
            time.sleep(0.001)  # Small sleep to reduce CPU usage
    
    def stop(self):
        """Stop the global mouse hook"""
        if self.running and self.hook_id:
            self.running = False
            ctypes.windll.user32.UnhookWindowsHookEx(self.hook_id)
            self.hook_id = None
            print("Global mouse hook removed")


def main():
    """Main function to run the mouse click tone generator"""
    if sys.platform != 'win32':
        print("This script only supports Windows for global mouse detection")
        sys.exit(1)
    
    print("Mouse Click Tone Generator")
    print("==========================")
    print("This script plays a tone on output 3/4 whenever you click your mouse anywhere.")
    print("Press Ctrl+C to exit.")
    
    # Create the tone generator
    tone_generator = MouseClickToneGenerator(amplitude=0.25)
    
    # Create the global mouse hook
    mouse_hook = GlobalMouseHook(tone_generator.play_tone)
    
    try:
        # Start the mouse hook
        mouse_hook.start()
        
        # Keep the main thread alive
        print("Mouse click detection active. Each dot (.) represents a detected click:")
        while True:
            time.sleep(1)
            
    except KeyboardInterrupt:
        print("\nExiting...")
    finally:
        # Clean up
        mouse_hook.stop()
        tone_generator.cleanup()


if __name__ == "__main__":
    main()

Mouse Click Tone Generator
This script plays a tone on output 3/4 whenever you click your mouse anywhere.
Press Ctrl+C to exit.
Found Output 3/4 device: ID 7 - Output 3/4 (2- Komplete Audio 6
Audio stream opened successfully on device 7
Tone generator initialized with frequency 1000Hz, amplitude 0.25

Tone generator cleaned up


Exception: Failed to install mouse hook

In [2]:
   self.hook_id = self.user32.SetWindowsHookExA(
            self.WH_MOUSE_LL,
            self.hook_proc,
            module_handle,
            0
        )
        
        if not self.hook_id:
            error_code = ctypes.get_last_error()
            raise Exception(f"Failed to install mouse hook. Error code: {error_code}")
        
        self.running = True
        print("Global mouse hook installed successfully")
        
        # Start message loop in a separate thread
        self.message_thread = threading.Thread(target=self._message_loop, daemon=True)
        self.message_thread.start()
        
    def _message_loop(self):
        """The Windows message loop required for hook processing"""
        msg = wintypes.MSG()
        
        # Process messages until told to stop
        while self.running:
            result = self.user32.GetMessageA(byref(msg), None, 0, 0)
            if result > 0:
                self.user32.TranslateMessage(byref(msg))
                self.user32.DispatchMessageA(byref(msg))
            time.sleep(0.001)  # Small sleep to reduce CPU usage
    
    def stop(self):
        """Stop the mouse hook"""
        if not self.running:
            return
            
        if self.hook_id:
            if not self.user32.UnhookWindowsHookEx(self.hook_id):
                error_code = ctypes.get_last_error()
                print(f"Warning: Failed to unhook mouse. Error code: {error_code}")
            self.hook_id = None
        
        self.running = False
        print("Global mouse hook removed")


class MouseClickToneApp:
    """GUI Application for the mouse click tone generator"""
    def __init__(self, root):
        self.root = root
        self.root.title("Mouse Click Tone Generator")
        self.root.geometry("400x300")
        self.root.resizable(False, False)
        
        # Set window icon (use system icon)
        try:
            self.root.iconbitmap(default="winlogo")
        except:
            pass
        
        # Create the tone generator
        self.tone_generator = MouseClickToneGenerator()
        
        # Create the mouse hook (but don't start it yet)
        self.mouse_hook = None
        
        # Create click counter
        self.click_count = 0
        
        # Track whether the hook is active
        self.hook_active = False
        
        # Create the UI
        self._create_ui()
        
        # Handle window closing
        self.root.protocol("WM_DELETE_WINDOW", self.on_closing)
        
        # Start the mouse hook by default
        self.toggle_hook()
    
    def _create_ui(self):
        """Create the user interface"""
        # Create main frame with padding
        frame = ttk.Frame(self.root, padding=10)
        frame.pack(fill=tk.BOTH, expand=True)
        
        # Title label
        title_label = ttk.Label(
            frame, 
            text="Mouse Click Tone Generator", 
            font=("Arial", 14, "bold")
        )
        title_label.pack(pady=(0, 10))
        
        # Status and controls
        controls_frame = ttk.LabelFrame(frame, text="Controls", padding=10)
        controls_frame.pack(fill=tk.X, pady=5)
        
        # Status indicator
        status_frame = ttk.Frame(controls_frame)
        status_frame.pack(fill=tk.X, pady=5)
        
        ttk.Label(status_frame, text="Status:").pack(side=tk.LEFT, padx=(0, 5))
        
        self.status_var = tk.StringVar(value="Inactive")
        status_label = ttk.Label(
            status_frame, 
            textvariable=self.status_var,
            font=("Arial", 10, "bold")
        )
        status_label.pack(side=tk.LEFT)
        
        # Click counter
        counter_frame = ttk.Frame(controls_frame)
        counter_frame.pack(fill=tk.X, pady=5)
        
        ttk.Label(counter_frame, text="Clicks detected:").pack(side=tk.LEFT, padx=(0, 5))
        
        self.counter_var = tk.StringVar(value="0")
        counter_label = ttk.Label(
            counter_frame, 
            textvariable=self.counter_var,
            width=8
        )
        counter_label.pack(side=tk.LEFT)
        
        # Tone test button
        test_button = ttk.Button(
            controls_frame,
            text="Test Tone",
            command=self._test_tone
        )
        test_button.pack(fill=tk.X, pady=5)
        
        # Start/Stop button
        self.toggle_button = ttk.Button(
            controls_frame,
            text="Start",
            command=self.toggle_hook
        )
        self.toggle_button.pack(fill=tk.X, pady=5)
        
        # Tone settings
        settings_frame = ttk.LabelFrame(frame, text="Tone Settings", padding=10)
        settings_frame.pack(fill=tk.X, pady=10)
        
        # Frequency
        freq_frame = ttk.Frame(settings_frame)
        freq_frame.pack(fill=tk.X, pady=3)
        
        ttk.Label(freq_frame, text="Frequency:").pack(side=tk.LEFT)
        
        self.freq_var = tk.IntVar(value=1000)
        freq_scale = ttk.Scale(
            freq_frame,
            from_=200,
            to=2000,
            variable=self.freq_var,
            command=self._update_frequency
        )
        freq_scale.pack(side=tk.LEFT, fill=tk.X, expand=True, padx=5)
        
        freq_label = ttk.Label(freq_frame, text="1000 Hz", width=8)
        freq_label.pack(side=tk.LEFT)
        
        def update_freq_label(*args):
            freq_label.config(text=f"{self.freq_var.get()} Hz")
        
        self.freq_var.trace_add("write", update_freq_label)
        
        # Volume (amplitude)
        vol_frame = ttk.Frame(settings_frame)
        vol_frame.pack(fill=tk.X, pady=3)
        
        ttk.Label(vol_frame, text="Volume:").pack(side=tk.LEFT)
        
        self.vol_var = tk.DoubleVar(value=0.25)
        vol_scale = ttk.Scale(
            vol_frame,
            from_=0.01,
            to=0.5,
            variable=self.vol_var,
            command=self._update_volume
        )
        vol_scale.pack(side=tk.LEFT, fill=tk.X, expand=True, padx=5)
        
        vol_label = ttk.Label(vol_frame, text="25%", width=8)
        vol_label.pack(side=tk.LEFT)
        
        def update_vol_label(*args):
            vol_percent = int(self.vol_var.get() * 100)
            vol_label.config(text=f"{vol_percent}%")
        
        self.vol_var.trace_add("write", update_vol_label)
        
        # Footer with instruction
        footer = ttk.Label(
            frame,
            text="This application will play a tone on output 3/4\n"
                 "whenever you click your mouse anywhere on screen.",
            justify=tk.CENTER
        )
        footer.pack(pady=10)
        
    def _test_tone(self):
        """Play a test tone"""
        self.tone_generator.play_tone()
    
    def _update_frequency(self, *args):
        """Update the tone frequency"""
        try:
            freq = self.freq_var.get()
            self.tone_generator.set_frequency(freq)
        except Exception as e:
            print(f"Error updating frequency: {e}")
    
    def _update_volume(self, *args):
        """Update the tone volume (amplitude)"""
        try:
            vol = self.vol_var.get()
            self.tone_generator.set_amplitude(vol)
        except Exception as e:
            print(f"Error updating volume: {e}")
    
    def on_mouse_click(self):
        """Called when a mouse click is detected"""
        # Update the counter
        self.click_count += 1
        self.counter_var.set(str(self.click_count))
        
        # Play the tone
        self.tone_generator.play_tone()
    
    def toggle_hook(self):
        """Toggle the mouse hook on/off"""
        if self.hook_active:
            # Stop the hook
            if self.mouse_hook:
                try:
                    self.mouse_hook.stop()
                    self.mouse_hook = None
                except Exception as e:
                    print(f"Error stopping mouse hook: {e}")
            
            self.hook_active = False
            self.status_var.set("Inactive")
            self.toggle_button.config(text="Start")
        else:
            # Start the hook
            try:
                self.mouse_hook = GlobalMouseHook(self.on_mouse_click)
                self.mouse_hook.start()
                self.hook_active = True
                self.status_var.set("Active")
                self.toggle_button.config(text="Stop")
            except Exception as e:
                msg = f"Error starting mouse hook: {e}"
                print(msg)
                self.status_var.set("Error")
                
                # Try a simpler approach if the hook fails
                self._try_fallback_approach()
    
    def _try_fallback_approach(self):
        """Try a simpler approach to detect mouse clicks if the hook fails"""
        messagebox = tk.messagebox.askyesno(
            "Mouse Hook Error",
            "Failed to install global mouse hook. This may be due to security settings.\n\n"
            "Would you like to test the tone instead?",
        )
        
        if messagebox:
            self._test_tone()
    
    def on_closing(self):
        """Handle window closing"""
        print("Shutting down...")
        
        # Stop the mouse hook if active
        if self.mouse_hook:
            try:
                self.mouse_hook.stop()
            except:
                pass
        
        # Cleanup tone generator
        if hasattr(self, 'tone_generator'):
            self.tone_generator.cleanup()
        
        # Destroy the window
        self.root.destroy()


def main():
    """Main function to run the application"""
    if sys.platform != 'win32':
        print("This script only supports Windows for global mouse detection")
        sys.exit(1)
    
    # Create the tkinter root window
    root = tk.Tk()
    
    # Create the application
    app = MouseClickToneApp(root)
    
    # Start the main loop
    root.mainloop()


if __name__ == "__main__":
    main()

NameError: name 'POINT' is not defined

In [4]:
import sounddevice as sd
import numpy as np
import threading
import time
import sys
import tkinter as tk
from tkinter import ttk, messagebox
import ctypes
from ctypes import wintypes, byref

# Configure sounddevice for low latency
sd.default.latency = 'low'

class MouseClickToneGenerator:
    """Class to generate and play tones on mouse clicks using output 3/4"""
    def __init__(self, frequency=1000, duration=0.1, sample_rate=44100, amplitude=0.25):
        self.frequency = frequency    # Tone frequency in Hz
        self.duration = duration      # Tone duration in seconds
        self.sample_rate = sample_rate
        self.amplitude = amplitude    # Reduced amplitude to be less intrusive
        self.output_34_id = None
        self.stream = None
        self.active = False  # Flag to track if we're currently playing
        
        # Generate the tone once and reuse it
        self.tone_data = self._generate_tone()
        
        # Find output 3/4 device
        self._find_output_device()
        
        # Try to open the stream once at initialization
        self._open_stream()
        
        print(f"Tone generator initialized with frequency {frequency}Hz, amplitude {amplitude}")
    
    def _generate_tone(self):
        """Generate the tone signal"""
        t = np.linspace(0, self.duration, int(self.sample_rate * self.duration), endpoint=False)
        tone = self.amplitude * np.sin(2 * np.pi * self.frequency * t).astype(np.float32)
        
        # Apply short fade in/out to prevent clicks
        fade_samples = int(0.01 * self.sample_rate)  # 10ms fade
        fade_in = np.linspace(0, 1, fade_samples)
        fade_out = np.linspace(1, 0, fade_samples)
        
        tone[:fade_samples] *= fade_in
        tone[-fade_samples:] *= fade_out
        
        # Create stereo version (for output to both channels in device)
        stereo_tone = np.column_stack((tone, tone))
        return stereo_tone
    
    def _find_output_device(self):
        """Find the Output 3/4 device"""
        try:
            devices = sd.query_devices()
            
            # First try to use the default output device as a fallback
            self.output_34_id = sd.default.device[1]
            
            # Then look for Output 3/4 devices
            for i, device in enumerate(devices):
                if device['max_output_channels'] > 0:
                    name = device['name'].lower()
                    # Look for output 3/4 devices
                    if "output" in name and "3/4" in name:
                        self.output_34_id = i
                        print(f"Found Output 3/4 device: ID {i} - {device['name']}")
                        return
                    # Also check for ASIO devices with 3/4 in the name
                    elif "asio" in name and "3/4" in name:
                        self.output_34_id = i
                        print(f"Found ASIO Output 3/4 device: ID {i} - {device['name']}")
                        return
            
            # If we didn't find a device with 3/4 explicitly, look for any 4-channel device
            for i, device in enumerate(devices):
                if device['max_output_channels'] >= 4:
                    self.output_34_id = i
                    print(f"Found multi-channel device: ID {i} - {device['name']}")
                    print("Using channels 3/4 on this device")
                    return
            
            print(f"Using default output device ID {self.output_34_id}")
        except Exception as e:
            print(f"Error finding output device: {e}")
    
    def _open_stream(self):
        """Open a persistent audio stream"""
        try:
            # Create a silent buffer for when not playing
            self.silent_buffer = np.zeros((1024, 2), dtype=np.float32)
            
            # Create the stream with a callback
            self.stream = sd.OutputStream(
                device=self.output_34_id,
                channels=2,
                callback=self._stream_callback,
                samplerate=self.sample_rate,
                blocksize=256,  # Small blocksize for low latency
                dtype='float32'
            )
            
            # Start the stream
            self.stream.start()
            print(f"Audio stream opened successfully on device {self.output_34_id}")
            
            # Create a position variable to track where we are in the tone
            self.position = 0
            
        except Exception as e:
            print(f"Error opening audio stream: {e}")
            self.stream = None
    
    def _stream_callback(self, outdata, frames, time, status):
        """Callback for the persistent audio stream"""
        if status:
            print(f"Stream callback status: {status}")
            
        if not self.active:
            # If not active, just fill with silence
            outdata.fill(0)
            return
            
        # We're playing the tone
        if self.position + frames > len(self.tone_data):
            # We've reached the end of the tone
            remaining = len(self.tone_data) - self.position
            if remaining > 0:
                outdata[:remaining] = self.tone_data[self.position:self.position+remaining]
            outdata[remaining:] = 0
            self.position = 0
            self.active = False  # Done playing
        else:
            # Still playing the tone
            outdata[:] = self.tone_data[self.position:self.position+frames]
            self.position += frames
    
    def play_tone(self):
        """Play the tone"""
        if self.stream is None or not self.stream.active:
            # Try to reopen the stream if it's not open
            self._open_stream()
            if self.stream is None:
                print("Cannot play tone: no active audio stream")
                return
        
        # Reset position and set active flag
        self.position = 0
        self.active = True
    
    def cleanup(self):
        """Clean up resources"""
        if self.stream:
            self.stream.stop()
            self.stream.close()
            self.stream = None
        print("Tone generator cleaned up")

    def set_frequency(self, freq):
        """Update the tone frequency"""
        self.frequency = freq
        self.tone_data = self._generate_tone()
        
    def set_amplitude(self, amp):
        """Update the tone amplitude"""
        self.amplitude = amp
        self.tone_data = self._generate_tone()


# Define required Windows API structures outside the class
class POINT(ctypes.Structure):
    _fields_ = [("x", ctypes.c_long), ("y", ctypes.c_long)]

class MSLLHOOKSTRUCT(ctypes.Structure):
    _fields_ = [
        ("pt", POINT),
        ("mouseData", wintypes.DWORD),
        ("flags", wintypes.DWORD),
        ("time", wintypes.DWORD),
        ("dwExtraInfo", ctypes.POINTER(ctypes.c_ulong))
    ]

class GlobalMouseHook:
    """Class to detect mouse clicks system-wide on Windows"""
    
    WH_MOUSE_LL = 14
    WM_LBUTTONDOWN = 0x0201
    WM_RBUTTONDOWN = 0x0204
    WM_MBUTTONDOWN = 0x0207
    
    def __init__(self, callback):
        self.callback = callback
        self.user32 = ctypes.WinDLL('user32', use_last_error=True)
        self.kernel32 = ctypes.WinDLL('kernel32', use_last_error=True)
        
        self.hook_id = None
        self.running = False
        
        # Define the callback prototype
        self.LowLevelMouseProc = ctypes.WINFUNCTYPE(
            ctypes.c_int, ctypes.c_int, wintypes.WPARAM, wintypes.LPARAM)
        
        # Create the actual callback function
        self.hook_proc = self.LowLevelMouseProc(self.hook_callback)
        
    def hook_callback(self, nCode, wParam, lParam):
        """The callback function called by Windows on mouse events"""
        if nCode >= 0:
            # Check for mouse button down events
            if wParam in (self.WM_LBUTTONDOWN, self.WM_RBUTTONDOWN, self.WM_MBUTTONDOWN):
                # Call the user callback
                self.callback()
                
        # Always call the next hook
        return self.user32.CallNextHookEx(self.hook_id, nCode, wParam, lParam)
    
    def start(self):
        """Start the mouse hook"""
        if self.running:
            return
            
        # Get module handle for the current process
        module_handle = self.kernel32.GetModuleHandleW(None)
        
        # Set the Windows hook
        self.hook_id = self.user32.SetWindowsHookExA(
            self.WH_MOUSE_LL,
            self.hook_proc,
            module_handle,
            0
        )
        
        if not self.hook_id:
            error_code = ctypes.get_last_error()
            raise Exception(f"Failed to install mouse hook. Error code: {error_code}")
        
        self.running = True
        print("Global mouse hook installed successfully")
        
        # Start message loop in a separate thread
        self.message_thread = threading.Thread(target=self._message_loop, daemon=True)
        self.message_thread.start()
        
    def _message_loop(self):
        """The Windows message loop required for hook processing"""
        msg = wintypes.MSG()
        
        # Process messages until told to stop
        while self.running:
            result = self.user32.GetMessageA(byref(msg), None, 0, 0)
            if result > 0:
                self.user32.TranslateMessage(byref(msg))
                self.user32.DispatchMessageA(byref(msg))
            time.sleep(0.001)  # Small sleep to reduce CPU usage
    
    def stop(self):
        """Stop the mouse hook"""
        if not self.running:
            return
            
        if self.hook_id:
            if not self.user32.UnhookWindowsHookEx(self.hook_id):
                error_code = ctypes.get_last_error()
                print(f"Warning: Failed to unhook mouse. Error code: {error_code}")
            self.hook_id = None
        
        self.running = False
        print("Global mouse hook removed")


class MinimalToneApp:
    """Minimal GUI for the mouse click tone generator"""
    def __init__(self, root):
        self.root = root
        self.root.title("Mouse Click Tone")
        self.root.geometry("250x150")
        self.root.resizable(False, False)
        
        # Try to make the window stay on top
        self.root.attributes("-topmost", True)
        
        # Create the tone generator
        self.tone_generator = MouseClickToneGenerator()
        
        # Create the mouse hook (but don't start it yet)
        self.mouse_hook = None
        
        # Track whether the hook is active
        self.hook_active = False
        
        # Create the UI
        self._create_ui()
        
        # Handle window closing
        self.root.protocol("WM_DELETE_WINDOW", self.on_closing)
        
        # Start the mouse hook by default
        self.toggle_hook()
    
    def _create_ui(self):
        """Create a minimal user interface"""
        # Create main frame with padding
        frame = ttk.Frame(self.root, padding=10)
        frame.pack(fill=tk.BOTH, expand=True)
        
        # Status indicator
        status_frame = ttk.Frame(frame)
        status_frame.pack(fill=tk.X, pady=5)
        
        ttk.Label(status_frame, text="Status:").pack(side=tk.LEFT, padx=(0, 5))
        
        self.status_var = tk.StringVar(value="Inactive")
        status_label = ttk.Label(
            status_frame, 
            textvariable=self.status_var,
            font=("Arial", 10, "bold")
        )
        status_label.pack(side=tk.LEFT)
        
        # Click counter
        counter_frame = ttk.Frame(frame)
        counter_frame.pack(fill=tk.X, pady=5)
        
        ttk.Label(counter_frame, text="Clicks:").pack(side=tk.LEFT, padx=(0, 5))
        
        self.counter_var = tk.StringVar(value="0")
        counter_label = ttk.Label(
            counter_frame, 
            textvariable=self.counter_var,
            width=8
        )
        counter_label.pack(side=tk.LEFT)
        
        # Test tone button
        test_button = ttk.Button(
            frame,
            text="Test Tone",
            command=self._test_tone
        )
        test_button.pack(fill=tk.X, pady=5)
        
        # Start/Stop button
        self.toggle_button = ttk.Button(
            frame,
            text="Start",
            command=self.toggle_hook
        )
        self.toggle_button.pack(fill=tk.X, pady=5)
    
    def _test_tone(self):
        """Play a test tone"""
        self.tone_generator.play_tone()
    
    def on_mouse_click(self):
        """Called when a mouse click is detected"""
        # Update the counter
        count = int(self.counter_var.get())
        count += 1
        self.counter_var.set(str(count))
        
        # Play the tone
        self.tone_generator.play_tone()
    
    def toggle_hook(self):
        """Toggle the mouse hook on/off"""
        if self.hook_active:
            # Stop the hook
            if self.mouse_hook:
                try:
                    self.mouse_hook.stop()
                    self.mouse_hook = None
                except Exception as e:
                    print(f"Error stopping mouse hook: {e}")
            
            self.hook_active = False
            self.status_var.set("Inactive")
            self.toggle_button.config(text="Start")
        else:
            # Start the hook
            try:
                self.mouse_hook = GlobalMouseHook(self.on_mouse_click)
                self.mouse_hook.start()
                self.hook_active = True
                self.status_var.set("Active")
                self.toggle_button.config(text="Stop")
            except Exception as e:
                msg = f"Error starting mouse hook: {e}"
                print(msg)
                self.status_var.set("Error")
                messagebox.showerror("Error", f"Failed to install mouse hook: {e}")
    
    def on_closing(self):
        """Handle window closing"""
        print("Shutting down...")
        
        # Stop the mouse hook if active
        if self.mouse_hook:
            try:
                self.mouse_hook.stop()
            except:
                pass
        
        # Cleanup tone generator
        if hasattr(self, 'tone_generator'):
            self.tone_generator.cleanup()
        
        # Destroy the window
        self.root.destroy()


class NoGuiToneApp:
    """Non-GUI version of the mouse click tone application"""
    def __init__(self):
        print("Starting Mouse Click Tone Generator (No GUI)")
        print("Press Ctrl+C to exit")
        
        # Create the tone generator
        self.tone_generator = MouseClickToneGenerator()
        
        # Create mouse hook
        self.mouse_hook = None
        self.click_count = 0
        
        # Start the hook
        try:
            self.mouse_hook = GlobalMouseHook(self.on_mouse_click)
            self.mouse_hook.start()
            print("Mouse hook active - detecting clicks system-wide")
        except Exception as e:
            print(f"Error starting mouse hook: {e}")
            return
        
        # Keep the program running
        try:
            while True:
                time.sleep(1)
        except KeyboardInterrupt:
            print("\nExiting...")
        finally:
            self.cleanup()
    
    def on_mouse_click(self):
        """Called when a mouse click is detected"""
        self.click_count += 1
        print(f"Click detected ({self.click_count})")
        
        # Play the tone
        self.tone_generator.play_tone()
    
    def cleanup(self):
        """Clean up resources"""
        if self.mouse_hook:
            try:
                self.mouse_hook.stop()
            except:
                pass
        
        if hasattr(self, 'tone_generator'):
            self.tone_generator.cleanup()


def main():
    """Main function"""
    if sys.platform != 'win32':
        print("This script only supports Windows for global mouse detection")
        sys.exit(1)
    
    # Check if we should run in no-gui mode
    if len(sys.argv) > 1 and sys.argv[1].lower() in ['-nogui', '--nogui', 'nogui']:
        NoGuiToneApp()
    else:
        # Create the tkinter root window
        root = tk.Tk()
        app = MinimalToneApp(root)
        root.mainloop()


if __name__ == "__main__":
    main()

Found Output 3/4 device: ID 7 - Output 3/4 (2- Komplete Audio 6
Audio stream opened successfully on device 7
Tone generator initialized with frequency 1000Hz, amplitude 0.25
Error starting mouse hook: Failed to install mouse hook. Error code: 126
Error starting mouse hook: Failed to install mouse hook. Error code: 126
Shutting down...
Tone generator cleaned up


In [5]:
import sounddevice as sd
import numpy as np
import threading
import time
import sys
import tkinter as tk
from tkinter import ttk
import ctypes
from ctypes import wintypes, byref

# Configure sounddevice for low latency
sd.default.latency = 'low'

class ToneGenerator:
    """Class to generate and play tones on mouse clicks using output 3/4"""
    def __init__(self, frequency=1000, duration=0.1, sample_rate=44100, amplitude=0.25):
        self.frequency = frequency    # Tone frequency in Hz
        self.duration = duration      # Tone duration in seconds
        self.sample_rate = sample_rate
        self.amplitude = amplitude    # Reduced to 0.25 to keep it less intrusive
        self.output_34_id = None
        self.stream = None
        self.active = False  # Flag to track if we're currently playing
        
        # Generate the tone once and reuse it
        self.tone_data = self._generate_tone()
        
        # Find output 3/4 device
        self._find_output_device()
        
        # Try to open the stream once at initialization
        self._open_stream()
        
        print(f"Tone generator initialized with frequency {frequency}Hz, amplitude {amplitude}")
    
    def _generate_tone(self):
        """Generate the tone signal"""
        t = np.linspace(0, self.duration, int(self.sample_rate * self.duration), endpoint=False)
        tone = self.amplitude * np.sin(2 * np.pi * self.frequency * t).astype(np.float32)
        
        # Apply short fade in/out to prevent clicks
        fade_samples = int(0.01 * self.sample_rate)  # 10ms fade
        fade_in = np.linspace(0, 1, fade_samples)
        fade_out = np.linspace(1, 0, fade_samples)
        
        tone[:fade_samples] *= fade_in
        tone[-fade_samples:] *= fade_out
        
        # Create stereo version (for output to both channels in device)
        stereo_tone = np.column_stack((tone, tone))
        return stereo_tone
    
    def _find_output_device(self):
        """Find the Output 3/4 device"""
        devices = sd.query_devices()
        
        # First try to use the default output device as a fallback
        self.output_34_id = sd.default.device[1]
        
        # Then look for Komplete Audio ASIO devices or similar
        for i, device in enumerate(devices):
            if device['max_output_channels'] > 0:
                name = device['name'].lower()
                # Look for output 3/4 devices
                if "output" in name and "3/4" in name:
                    self.output_34_id = i
                    print(f"Found Output 3/4 device: ID {i} - {device['name']}")
                    return
        
        print(f"Using default output device ID {self.output_34_id}")
    
    def _open_stream(self):
        """Open a persistent audio stream"""
        try:
            # Create a silent buffer for when not playing
            self.silent_buffer = np.zeros((1024, 2), dtype=np.float32)
            
            # Create the stream with a callback
            self.stream = sd.OutputStream(
                device=self.output_34_id,
                channels=2,
                callback=self._stream_callback,
                samplerate=self.sample_rate,
                blocksize=256,  # Small blocksize for low latency
                dtype='float32'
            )
            
            # Start the stream
            self.stream.start()
            print(f"Audio stream opened successfully on device {self.output_34_id}")
            
            # Create a position variable to track where we are in the tone
            self.position = 0
            
        except Exception as e:
            print(f"Error opening audio stream: {e}")
            self.stream = None
    
    def _stream_callback(self, outdata, frames, time, status):
        """Callback for the persistent audio stream"""
        if status:
            print(f"Stream callback status: {status}")
            
        if not self.active:
            # If not active, just fill with silence
            outdata.fill(0)
            return
            
        # We're playing the tone
        if self.position + frames > len(self.tone_data):
            # We've reached the end of the tone
            remaining = len(self.tone_data) - self.position
            if remaining > 0:
                outdata[:remaining] = self.tone_data[self.position:self.position+remaining]
            outdata[remaining:] = 0
            self.position = 0
            self.active = False  # Done playing
        else:
            # Still playing the tone
            outdata[:] = self.tone_data[self.position:self.position+frames]
            self.position += frames
    
    def play_tone(self):
        """Play the tone"""
        if self.stream is None or not self.stream.active:
            # Try to reopen the stream if it's not open
            self._open_stream()
            if self.stream is None:
                print("Cannot play tone: no active audio stream")
                return
        
        # Reset position and set active flag
        self.position = 0
        self.active = True
    
    def cleanup(self):
        """Clean up resources"""
        if self.stream:
            self.stream.stop()
            self.stream.close()
            self.stream = None
        print("Tone generator cleaned up")

    def set_frequency(self, freq):
        """Update the tone frequency"""
        self.frequency = freq
        self.tone_data = self._generate_tone()
        
    def set_amplitude(self, amp):
        """Update the tone amplitude"""
        self.amplitude = amp
        self.tone_data = self._generate_tone()


class MouseClickDetector:
    """Class to detect mouse clicks using polling approach"""
    def __init__(self, callback):
        self.callback = callback
        self.running = False
        self.poll_thread = None
        self.poll_interval = 0.005  # 5ms polling interval
        
        # Track previous button states
        self.last_left = False
        self.last_right = False
        self.last_middle = False
    
    def _get_button_states(self):
        """Get current state of mouse buttons using GetAsyncKeyState"""
        user32 = ctypes.windll.user32.GetAsyncKeyState
        # VK_LBUTTON = 0x01, VK_RBUTTON = 0x02, VK_MBUTTON = 0x04
        left = (user32(0x01) & 0x8000) != 0
        right = (user32(0x02) & 0x8000) != 0
        middle = (user32(0x04) & 0x8000) != 0
        return left, right, middle
    
    def _poll_thread_func(self):
        """Thread function to continuously poll for mouse clicks"""
        try:
            while self.running:
                # Get current button states
                left, right, middle = self._get_button_states()
                
                # Check for button press events (transition from not pressed to pressed)
                if left and not self.last_left:
                    self.callback(0)  # 0 = left button
                if right and not self.last_right:
                    self.callback(1)  # 1 = right button
                if middle and not self.last_middle:
                    self.callback(2)  # 2 = middle button
                
                # Update previous states
                self.last_left = left
                self.last_right = right
                self.last_middle = middle
                
                # Sleep a bit to reduce CPU usage
                time.sleep(self.poll_interval)
        except Exception as e:
            print(f"Error in polling thread: {e}")
            self.running = False
    
    def start(self):
        """Start polling for mouse clicks"""
        if self.running:
            return
        
        self.running = True
        self.poll_thread = threading.Thread(target=self._poll_thread_func, daemon=True)
        self.poll_thread.start()
        print("Mouse click polling started")
    
    def stop(self):
        """Stop polling for mouse clicks"""
        self.running = False
        if self.poll_thread:
            self.poll_thread.join(timeout=1.0)
            self.poll_thread = None
        print("Mouse click polling stopped")


class ClickToneApp:
    """GUI Application for mouse click tone generator"""
    def __init__(self, root):
        self.root = root
        self.root.title("Mouse Click Tone")
        self.root.geometry("250x150")
        self.root.resizable(False, False)
        
        # Try to stay on top
        try:
            self.root.attributes("-topmost", True)
        except:
            pass
        
        # Create tone generator
        self.tone_generator = ToneGenerator()
        
        # Create mouse detector
        self.mouse_detector = None
        self.active = False
        self.click_count = 0
        
        # Create UI
        self._create_ui()
        
        # Handle window closing
        self.root.protocol("WM_DELETE_WINDOW", self.on_closing)
        
        # Start automatically
        self.toggle_detection()
    
    def _create_ui(self):
        """Create a minimal UI"""
        frame = ttk.Frame(self.root, padding=10)
        frame.pack(fill=tk.BOTH, expand=True)
        
        # Status indicator
        status_frame = ttk.Frame(frame)
        status_frame.pack(fill=tk.X, pady=5)
        ttk.Label(status_frame, text="Status:").pack(side=tk.LEFT, padx=(0, 5))
        self.status_var = tk.StringVar(value="Inactive")
        ttk.Label(status_frame, textvariable=self.status_var, font=("Arial", 10, "bold")).pack(side=tk.LEFT)
        
        # Click counter
        counter_frame = ttk.Frame(frame)
        counter_frame.pack(fill=tk.X, pady=5)
        ttk.Label(counter_frame, text="Clicks:").pack(side=tk.LEFT, padx=(0, 5))
        self.counter_var = tk.StringVar(value="0")
        ttk.Label(counter_frame, textvariable=self.counter_var).pack(side=tk.LEFT)
        
        # Test and toggle buttons
        ttk.Button(frame, text="Test Tone", command=self.test_tone).pack(fill=tk.X, pady=5)
        self.toggle_button = ttk.Button(frame, text="Start", command=self.toggle_detection)
        self.toggle_button.pack(fill=tk.X, pady=5)
    
    def on_mouse_click(self, button_idx):
        """Called when a mouse click is detected"""
        self.click_count += 1
        self.counter_var.set(str(self.click_count))
        
        # Play the tone
        self.tone_generator.play_tone()
    
    def test_tone(self):
        """Play a test tone"""
        self.tone_generator.play_tone()
    
    def toggle_detection(self):
        """Toggle mouse click detection on/off"""
        if self.active:
            # Stop detection
            if self.mouse_detector:
                self.mouse_detector.stop()
                self.mouse_detector = None
            
            self.active = False
            self.status_var.set("Inactive")
            self.toggle_button.config(text="Start")
        else:
            # Start detection
            self.mouse_detector = MouseClickDetector(self.on_mouse_click)
            self.mouse_detector.start()
            
            self.active = True
            self.status_var.set("Active")
            self.toggle_button.config(text="Stop")
    
    def on_closing(self):
        """Handle window closing"""
        print("Shutting down...")
        
        # Stop mouse detection
        if self.mouse_detector:
            self.mouse_detector.stop()
        
        # Clean up tone generator
        self.tone_generator.cleanup()
        
        # Destroy window
        self.root.destroy()


class NoGuiClickToneApp:
    """Non-GUI version of the mouse click tone application"""
    def __init__(self):
        print("Starting Mouse Click Tone Generator (No GUI)")
        print("Press Ctrl+C to exit")
        
        # Create tone generator
        self.tone_generator = ToneGenerator()
        
        # Play test tone
        print("Playing test tone...")
        self.tone_generator.play_tone()
        time.sleep(0.5)
        
        # Create mouse detector
        self.mouse_detector = None
        self.click_count = 0
        
        # Start detection
        try:
            self.mouse_detector = MouseClickDetector(self.on_mouse_click)
            self.mouse_detector.start()
            print("Mouse detection active - detecting clicks system-wide")
        except Exception as e:
            print(f"Error starting mouse detection: {e}")
            return
        
        # Keep running
        try:
            while True:
                time.sleep(1)
        except KeyboardInterrupt:
            print("\nExiting...")
        finally:
            self.cleanup()
    
    def on_mouse_click(self, button_idx):
        """Called when a mouse click is detected"""
        self.click_count += 1
        button_name = ["left", "right", "middle"][button_idx]
        print(f"Click detected: {button_name} button ({self.click_count} total)")
        
        # Play the tone
        self.tone_generator.play_tone()
    
    def cleanup(self):
        """Clean up resources"""
        if self.mouse_detector:
            self.mouse_detector.stop()
        
        if hasattr(self, 'tone_generator'):
            self.tone_generator.cleanup()


def main():
    """Main function"""
    if sys.platform != 'win32':
        print("This script only supports Windows")
        sys.exit(1)
    
    # Check if we should run in no-gui mode
    if len(sys.argv) > 1 and sys.argv[1].lower() in ['-nogui', '--nogui', 'nogui']:
        NoGuiClickToneApp()
    else:
        # Create tkinter root window
        root = tk.Tk()
        app = ClickToneApp(root)
        root.mainloop()


if __name__ == "__main__":
    main()

Found Output 3/4 device: ID 7 - Output 3/4 (2- Komplete Audio 6
Audio stream opened successfully on device 7
Tone generator initialized with frequency 1000Hz, amplitude 0.25
Mouse click polling started
Shutting down...
Mouse click polling stopped
Tone generator cleaned up
